In [ ]:
orders = data["orders"]
paid = orders[orders["Financial Status"] == "paid"] if "Financial Status" in orders.columns else orders

st.subheader("Financial Overview")

discount_pct = paid["Has Discount"].mean() * 100 if "Has Discount" in paid.columns else 0
total_discount = paid["Discount Amount"].sum() if "Discount Amount" in paid.columns else 0
cod_share = (orders["Payment Type"] == "COD").mean() * 100 if "Payment Type" in orders.columns else 0
optin_rate = (orders["Accepts Marketing"] == "yes").mean() * 100 if "Accepts Marketing" in orders.columns else 0

c1, c2, c3, c4 = st.columns(4)
with c1:
    kpi_card("Orders Using Discount", f"{discount_pct:.1f}%")
with c2:
    kpi_card("Total Discount Given", f"₹{total_discount:,.0f}")
with c3:
    kpi_card("COD Share", f"{cod_share:.1f}%")
with c4:
    kpi_card("Marketing Opt-in", f"{optin_rate:.1f}%")

In [ ]:
col1, col2 = st.columns(2)
with col1:
    if "Discount Category" in paid.columns:
        disc = paid["Discount Category"].value_counts().reset_index()
        disc.columns = ["Category", "Orders"]
        fig = px.bar(disc, x="Orders", y="Category", orientation="h", title="Paid Orders by Discount Category",
                      color_discrete_sequence=[ACCENT1])
        fig.update_layout(yaxis=dict(autorange="reversed"))
        st.plotly_chart(fig, width='stretch')

with col2:
    if "Payment Type" in orders.columns:
        pay_summary = orders.groupby("Payment Type").agg(
            Orders=("Name", "nunique"),
            Cancellation_Rate=("Is Cancelled", lambda x: x.mean() * 100),
            RTO_Rate=("Any RTO", lambda x: x.mean() * 100),
        ).round(2).reset_index()
        fig = go.Figure()
        fig.add_bar(x=pay_summary["Payment Type"], y=pay_summary["Cancellation_Rate"], name="Cancellation Rate %", marker_color=ACCENT2)
        fig.add_bar(x=pay_summary["Payment Type"], y=pay_summary["RTO_Rate"], name="RTO Rate %", marker_color=DANGER)
        fig.update_layout(barmode="group", title="Cancellation & RTO Rate by Payment Type")
        st.plotly_chart(fig, width='stretch')

In [ ]:
# Cancellation trend
if "Order Month" in orders.columns:
    st.subheader("Cancellation Rate Trend")
    cancel_trend = orders.groupby("Order Month")["Is Cancelled"].mean().reset_index()
    cancel_trend["Is Cancelled"] *= 100
    cancel_trend["Order Month"] = cancel_trend["Order Month"].astype(str)
    fig = px.line(cancel_trend, x="Order Month", y="Is Cancelled", markers=True,
                   title="Monthly Cancellation Rate (%)")
    fig.update_traces(line_color=DANGER, fill="tozeroy", fillcolor="rgba(217,105,95,0.10)")
    fig.update_layout(yaxis_title="Cancellation Rate %")
    st.plotly_chart(fig, width='stretch')

In [ ]:
# Revenue waterfall
if all(c in paid.columns for c in ["Subtotal", "Discount Amount", "Taxes", "Shipping", "Total"]):
    st.subheader("Revenue Waterfall (Paid Orders)")
    subtotal = paid["Subtotal"].sum()
    discount = paid["Discount Amount"].sum()
    taxes = paid["Taxes"].sum()
    shipping = paid["Shipping"].sum()
    total = paid["Total"].sum()
    refunds = paid["Refunded Amount"].sum() if "Refunded Amount" in paid.columns else 0

    fig = go.Figure(go.Waterfall(
        orientation="v",
        measure=["absolute", "relative", "relative", "relative", "total", "relative", "total"],
        x=["Gross Subtotal", "Discounts", "Taxes", "Shipping", "Total", "Refunds", "Net Revenue"],
        y=[subtotal, -discount, taxes, shipping, 0, -refunds, 0],
        connector={"line": {"color": "#2A2D38"}},
        increasing={"marker": {"color": SUCCESS}},
        decreasing={"marker": {"color": DANGER}},
        totals={"marker": {"color": ACCENT1}},
    ))
    fig.update_layout(title="Revenue Breakdown")
    st.plotly_chart(fig, width='stretch')

In [ ]:
# Marketing channel attribution (UTM source/medium/campaign, extracted from Note Attributes)
if "UTM Source" in orders.columns and orders["UTM Source"].notna().any():
    st.markdown("---")
    st.subheader("Marketing Channel Attribution")

    utm = orders.dropna(subset=["UTM Source"]).copy()

    if "Source Channel" in utm.columns and "Source Platform" in utm.columns:
        granularity = st.radio(
            "Granularity", ["Consolidated Channel", "Individual Platform"],
            horizontal=True, key="marketing_channel_granularity",
            help=(
                "Consolidated Channel groups Facebook + Instagram together as 'Meta "
                "(Facebook/Instagram)'. Individual Platform keeps Facebook, Instagram, "
                "Google, Shopify Email, etc. as separate rows -- each still merges its "
                "own spelling variants (e.g. all of 'facebook'/'FB'/'fb'/'Meta_fb' -> "
                "'Facebook'; all of 'Meta_ig'/'instagram'/'ig'/'instagarm' -> 'Instagram')."
            ),
        )
        group_col = "Source Channel" if granularity == "Consolidated Channel" else "Source Platform"

        chan_summary = utm.groupby(group_col).agg(
            Orders=("Name", "nunique"),
            Revenue=("Total", "sum"),
            RTO_Rate=("Any RTO", lambda x: x.mean() * 100),
        ).round(2).sort_values("Orders", ascending=False)

        col1, col2 = st.columns(2)
        with col1:
            fig = px.bar(chan_summary.reset_index(), x="Orders", y=group_col, orientation="h",
                          title=f"Orders by {granularity}", color_discrete_sequence=[ACCENT1])
            fig.update_layout(yaxis=dict(autorange="reversed"))
            st.plotly_chart(fig, width="stretch")
        with col2:
            fig = px.bar(chan_summary.reset_index(), x="Revenue", y=group_col, orientation="h",
                          title=f"Revenue by {granularity}", color_discrete_sequence=[ACCENT2])
            fig.update_layout(yaxis=dict(autorange="reversed"))
            st.plotly_chart(fig, width="stretch")

        st.dataframe(chan_summary, width="stretch")
        st.caption(
            "Raw `utm_source` values from ad platforms / the checkout app (GoKwik) / affiliate "
            "tools were never standardized -- e.g. 'facebook', 'FB', 'fb', 'Meta', 'Meta_fb', "
            "'Meta_ig' and 'META' are all really Meta (Facebook/Instagram) traffic, and "
            "'Uniqoemedia', 'kwikengage', 'adzck', etc. are affiliate/influencer networks. "
            "Switch to 'Individual Platform' to see Facebook vs Instagram vs Google vs "
            "Shopify Email etc. broken out separately (each still de-duplicated internally)."
        )

        with st.expander("Show raw UTM Source values within a group"):
            sel_group = st.selectbox("Group", chan_summary.index.tolist(), key="raw_source_group")
            raw_summary = utm[utm[group_col] == sel_group].groupby("UTM Source").agg(
                Orders=("Name", "nunique"),
                Revenue=("Total", "sum"),
                RTO_Rate=("Any RTO", lambda x: x.mean() * 100),
            ).round(2).sort_values("Orders", ascending=False)
            st.dataframe(raw_summary, width="stretch")
    else:
        utm_summary = utm.groupby("UTM Source").agg(
            Orders=("Name", "nunique"),
            Revenue=("Total", "sum"),
            RTO_Rate=("Any RTO", lambda x: x.mean() * 100),
        ).round(2).sort_values("Orders", ascending=False).head(12)
        st.dataframe(utm_summary, width="stretch")

    coverage = utm["Name"].nunique() / orders["Name"].nunique() * 100
    st.caption(
        f"`utm_source` was extracted from the order's Note Attributes (set by ad platforms / "
        f"checkout apps like GoKwik) and is available for {coverage:.0f}% of orders -- the rest "
        "are likely direct visits, older orders, or checkouts that didn't carry UTM parameters."
    )

In [ ]:
# Paid vs Organic traffic + Top campaigns
col1, col2 = st.columns([1, 1.6])
with col1:
    if "Traffic Type" in utm.columns:
        traffic = utm["Traffic Type"].value_counts().reset_index()
        traffic.columns = ["Traffic Type", "Orders"]
        fig = px.pie(traffic, names="Traffic Type", values="Orders", hole=0.55,
                      title="Paid vs Organic Traffic (orders with UTM data)",
                      color_discrete_sequence=PALETTE)
        st.plotly_chart(fig, width="stretch")
        st.caption(
            "Classified from `utm_medium`: 'paid'/'cpc'/'banner' -> Paid, "
            "'organic'/'instagram_*'/'bio' -> Organic Social, 'product_sync'/'feed' -> "
            "Catalog/Feed, 'automation' -> Automation/Retention, everything else -> Other."
        )

with col2:
    if "UTM Campaign" in utm.columns and utm["UTM Campaign"].notna().any():
        camp = utm.dropna(subset=["UTM Campaign"]).copy()
        camp_summary = camp.groupby("UTM Campaign").agg(
            Orders=("Name", "nunique"),
            Revenue=("Total", "sum"),
            RTO_Rate=("Any RTO", lambda x: x.mean() * 100),
        ).round(2).sort_values("Orders", ascending=False).head(12)
        st.markdown("**Top Campaigns by Orders**")
        st.dataframe(camp_summary, width="stretch")
        st.caption(
            "`utm_campaign` values are a mix of named campaigns (e.g. 'Catalys', 'Ignite "
            "Series') and raw Facebook campaign IDs -- the IDs would need to be matched "
            "against your Ads Manager to know what they were."
        )

In [ ]:
# Vendor / source performance
if "Vendor" in orders.columns:
    st.subheader("Vendor / Ad-Creative Performance")
    vendor_summary = orders.groupby("Vendor").agg(
        Orders=("Name", "nunique"),
        Paid_Rate=("Financial Status", lambda x: (x == "paid").mean() * 100) if "Financial Status" in orders.columns else ("Name", "count"),
        Avg_Order_Value=("Total", "mean"),
        Cancellation_Rate=("Is Cancelled", lambda x: x.mean() * 100),
        RTO_Rate=("Any RTO", lambda x: x.mean() * 100),
    ).round(2)
    vendor_summary = vendor_summary[vendor_summary["Orders"] >= max(10, len(orders) // 1000)].sort_values("Orders", ascending=False)
    st.dataframe(vendor_summary, width='stretch')